# Engine corrections (02a)

Stage 02a made five changes to the engine. One of them - the caution
process - was decided and implemented first, and Parts 1 to 7 of this
notebook are the evidence behind that decision's `Verified:` line. The
other four are the ones the stage blueprint lists, and they arrive from
Part 9 onwards.

**What Parts 1 to 7 check, and what they do not.** They reimplement both
caution processes from their descriptions, in about sixty lines, and check
the statistical claims against that reimplementation. That is deliberately
*not* the same as testing `src/endurance`. Two independent implementations
agreeing is a much stronger statement than one implementation agreeing with
itself, and it is the only way to catch the class of error 02a was written
about - where the code and the thing it was meant to compute had drifted
apart without either looking wrong on its own.

Part 8 closes that loop against the engine. Everything before it runs with
nothing but numpy.

**Parts 9 onwards take the four remaining corrections apart one at a
time**: per-car random streams, a regulation pit layer, field compression
with wave-arounds, and traffic that responds to current pace. Each is
switchable off through `Compat`, which is what makes "here is what the
change did" a claim you can check rather than one you have to accept.

**Findings this stage produced that no decision document contains** are
collected at the end rather than buried: the seed-noise caveat is pinned to
an operating point the corrected engine may no longer sit at; the stated
reason for superseding decision 17 does not survive the check that decision
17's replacement was verified with; the blueprint's claim that the WEC pit
lane stays open under caution is wrong; and the traffic correction, though
necessary, does almost nothing to a field that pits in lockstep.

### Setup

Nothing from `src/endurance` until Part 8, so this section runs standalone.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

DURATION_S = 24 * 3600.0          # the 24-hour anchor races
N_SEEDS    = 3000                 # matches the seed count 02a reports

# Roughly eight episodes in a 24-hour race, per 02a's "Consequences".
def mean_dur_for(share, n_episodes=8):
    return share * DURATION_S / n_episodes

## Part 1 - the two processes, side by side

The alternating process is the one 02a adopts: exponential green gaps,
exponential episodes, and a caution only able to start when none is running.
The green mean is not a free parameter - for an alternating renewal process
the stationary occupancy is

$$\text{rate} = \frac{\bar{d}}{\bar{d} + \bar{g}}$$

so fixing the rate and the episode mean pins the green mean. That single
line is what makes the realised share come out where it was asked to.

Both functions return drawn lengths alongside the episodes. The distinction
matters: an episode running past the chequered flag is clipped, so measuring
episode length on the timeline censors the longest ones and drags any shape
statistic down for reasons that have nothing to do with the model.

In [2]:
def draw_alternating(rng, rate, mean_dur_s, duration_s=DURATION_S,
                     stationary_start=True):
    """02a's process. Non-overlapping by construction."""
    mean_green_s = mean_dur_s * (1.0 - rate) / rate
    episodes, drawn = [], []
    t = 0.0

    if stationary_start and rng.random() < rate:
        # Memorylessness earns this: the residual of an episode already in
        # progress is itself Exp(mean_dur), so opening under caution needs
        # no special case and no separate parameter.
        length = rng.exponential(mean_dur_s)
        drawn.append(length)
        episodes.append((0.0, min(length, duration_s)))
        t = length

    while t < duration_s:
        t += rng.exponential(mean_green_s)
        if t >= duration_s:
            break
        length = rng.exponential(mean_dur_s)
        drawn.append(length)
        episodes.append((t, min(t + length, duration_s)))
        t += length

    return episodes, drawn


def draw_legacy_merged(rng, rate, mean_dur_s, duration_s=DURATION_S):
    """The pre-02a draw: n episodes at uniform starts, overlaps collapsed."""
    n = int(round(rate * duration_s / mean_dur_s))
    starts  = rng.random(n) * duration_s
    lengths = rng.exponential(mean_dur_s, size=n)
    raw = sorted(zip(starts, np.minimum(starts + lengths, duration_s)))

    merged = []
    for s, e in raw:
        if merged and s <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else:
            merged.append((s, e))
    return merged


def draw_rejection(rng, rate, mean_dur_s, duration_s=DURATION_S,
                   max_attempts=100_000):
    """Decision 17: redraw the whole configuration until nothing overlaps."""
    n = int(round(rate * duration_s / mean_dur_s))
    for attempt in range(1, max_attempts + 1):
        starts  = rng.random(n) * duration_s
        lengths = rng.exponential(mean_dur_s, size=n)
        ep = sorted(zip(starts, starts + lengths))
        if ep[-1][1] <= duration_s and all(ep[i][1] <= ep[i + 1][0]
                                           for i in range(n - 1)):
            return ep, attempt
    return None, max_attempts

In [3]:
def share(episodes, duration_s=DURATION_S):
    return sum(e - s for s, e in episodes) / duration_s

def lengths_of(episodes):
    return np.array([e - s for s, e in episodes])

def cv(x):
    x = np.asarray(x, dtype=float)
    return x.std(ddof=1) / x.mean()

def touching(episodes, tol=1e-9):
    return any(episodes[i][1] >= episodes[i + 1][0] - tol
               for i in range(len(episodes) - 1))

def interior(episodes, duration_s=DURATION_S, tol=1e-6):
    """Episodes clipped by neither end of the race - the uncensored ones."""
    return [(s, e) for s, e in episodes if s > tol and e < duration_s - tol]

master = np.random.default_rng(20260804)
seeds  = lambda n: master.integers(2**63, size=n)

## Part 2 - the three claims 02a rests on

Realised share unbiased at 0.05, 0.18 and 0.30; episode-length cv of 1.00;
no two episodes touching.

**A note on how the cv is measured, because the obvious way is wrong.** The
tempting version computes a cv per race and averages. With about eight
episodes a race the cv estimator is biased low by roughly a tenth *whatever
the underlying distribution is*, so that statistic reads about 0.89 for a
perfectly exponential process and cannot tell the processes apart. Pooling
every drawn episode across all seeds and taking one cv removes the bias.
This is worth stating because the merging draw's cv of 1.09 and the
alternating draw's 1.00 are only about a tenth apart, which is exactly the
size of the artefact.

In [4]:
rows = []
for target in (0.05, 0.18, 0.30):
    mean_dur = mean_dur_for(0.30)      # one episode scale across all three
    shares, pooled, counts, overlaps = [], [], [], 0

    for s in seeds(N_SEEDS):
        ep, drawn = draw_alternating(np.random.default_rng(s), target, mean_dur)
        shares.append(share(ep))
        pooled.extend(drawn)
        counts.append(len(ep))
        overlaps += touching(ep)

    shares = np.array(shares)
    rows.append({
        "target":       target,
        "realised":     round(shares.mean(), 4),
        "bias":         round(shares.mean() - target, 4),
        "mc_se":        round(shares.std() / np.sqrt(N_SEEDS), 4),
        "sd_seed":      round(shares.std(), 4),
        "cv_drawn":     round(cv(pooled), 3),
        "mean_episodes": round(np.mean(counts), 1),
        "races_with_touching_episodes": overlaps,
    })

pd.DataFrame(rows)

,target,realised,bias,mc_se,sd_seed,cv_drawn,mean_episodes,races_with_touching_episodes
0,0.05,0.0500,0.0000,0.0010,0.0562,1.005,1.4,0
1,0.18,0.1795,-0.0005,0.0017,0.0909,0.994,5.0,0
2,0.30,0.3001,0.0001,0.0019,0.1015,0.997,8.3,0


Unbiased at all three rates - every bias is inside its own Monte Carlo
standard error. The cv sits on 1.00 and not one race in nine thousand
produced episodes that touch, which is what "non-overlapping by
construction" should look like when it is true.

## Part 3 - why the stationary start is not a detail

02a starts the process in its stationary state: with probability
`caution_rate` the race opens under caution. Without it the timeline always
begins green, and the process needs a few hours to forget that it did.

The size of the loss is predictable. The two-state chain relaxes at
$\lambda = 1/\bar{d} + 1/\bar{g}$, so the deficit is about
$\text{rate} \times \lambda^{-1} / T$ - the share the race is missing,
for as long as it takes to forget where it started, over the length of the
race.

In [5]:
N_C = 20_000       # this comparison is a fraction of a point, so it needs seeds
rows = []

for target in (0.18, 0.30):
    mean_dur   = mean_dur_for(0.30)
    mean_green = mean_dur * (1 - target) / target
    tau        = 1.0 / (1.0 / mean_dur + 1.0 / mean_green)

    out = {}
    for stationary in (True, False):
        sh = [share(draw_alternating(np.random.default_rng(s), target,
                                     mean_dur, stationary_start=stationary)[0])
              for s in seeds(N_C)]
        out[stationary] = np.mean(sh)

    rows.append({
        "target":            target,
        "stationary_start":  round(out[True], 4),
        "green_start":       round(out[False], 4),
        "deficit":           round(out[True] - out[False], 4),
        "predicted_deficit": round(target * tau / DURATION_S, 4),
    })

pd.DataFrame(rows)

,target,stationary_start,green_start,deficit,predicted_deficit
0,0.18,0.1785,0.1744,0.0042,0.0055
1,0.30,0.2996,0.2924,0.0072,0.0079


About seven tenths of a point at the higher rate, four at the lower, both
close to the analytic prediction. 02a calls it "about a point light", which
is the right order and, at the rate the engine actually runs, close to
exact.

## Part 4 - what the merging draw was doing

Two separate faults, and 02a is right that they compound rather than
cancel.

The coverage loss has a closed form. Uniform starts make the episodes a
Poisson-like coverage process, so the expected union is
$1 - e^{-r}$ against a drawn total of $r$ - a shortfall that grows with the
rate, which is why 02a reports 14% at the old settings and 18% at corrected
ones. The simulated figure comes in slightly below the closed form because
episodes are also clipped at the chequered flag, which the closed form does
not model.

The shape damage is the more serious one for 02b: the union of overlapping
exponentials is not exponential, and merging is precisely the operation that
destroys the memorylessness 02b's benchmark rests on.

In [6]:
rows = []
for target in (0.30, 0.407):
    mean_dur = mean_dur_for(0.30)
    shares, pooled = [], []

    for s in seeds(N_SEEDS):
        ep = draw_legacy_merged(np.random.default_rng(s), target, mean_dur)
        shares.append(share(ep))
        pooled.extend(lengths_of(interior(ep)))     # uncensored only

    realised = np.mean(shares)
    rows.append({
        "target":            target,
        "realised":          round(realised, 4),
        "shortfall_pct":     round((realised - target) / target * 100, 1),
        "closed_form_1_minus_exp": round(1 - np.exp(-target), 4),
        "cv_merged":         round(cv(pooled), 3),
    })

pd.DataFrame(rows)

,target,realised,shortfall_pct,closed_form_1_minus_exp,cv_merged
0,0.300,0.2541,-15.3,0.2592,1.059
1,0.407,0.3295,-19.0,0.3344,1.076


A cv of about 1.06-1.08 against the 1.00 an exponential gives. 02a reports
1.09; the difference is which episodes are counted, and either way the
conclusion is the same one. Note how small the signal is in absolute terms -
this is the artefact that a per-race cv, biased low by a tenth, would have
hidden completely.

## Part 5 - the units chain, end to end

02a's headline: a calibrated 0.30 caution *lap* share came out of the engine
at 0.19. That number is the product of three steps, and reconstructing it
confirms the diagnosis rather than merely restating it.

A caution lap occupies `caution_pace_multiplier` times the wall clock of a
green one, so a lap share $f$ and a time share $t$ are related by

$$t = \frac{fm}{fm + (1-f)}$$

Step one, a 0.30 lap share is really a time share of roughly 0.39-0.41.
Step two, the engine reads 0.30 as a time target anyway and the merge
delivers $1 - e^{-0.30} = 0.259$ of it. Step three, notebook 01 Part 6
reports the result back as a lap share.

In [7]:
lap_to_time = lambda f, m: f * m / (f * m + (1 - f))
time_to_lap = lambda t, m: (t / m) / (t / m + (1 - t))

legacy_time = 1 - np.exp(-0.30)     # 0.30 misread as a time target, then merged

pd.DataFrame([{
    "caution_pace_multiplier": m,
    "true_time_share":         round(lap_to_time(0.30, m), 4),
    "legacy_realised_time":    round(legacy_time, 4),
    "reported_as_lap_share":   round(time_to_lap(legacy_time, m), 4),
} for m in (1.4, 1.5, 1.6, 1.7, 2.0)])

,caution_pace_multiplier,true_time_share,legacy_realised_time,reported_as_lap_share
0,1.4,0.3750,0.2592,0.1999
1,1.5,0.3913,0.2592,0.1891
2,1.6,0.4068,0.2592,0.1794
3,1.7,0.4215,0.2592,0.1707
4,2.0,0.4615,0.2592,0.1489


The chain closes at a multiplier between 1.4 and 1.5, which reproduces 02a's
0.19 to two decimal places. It also implies the corrected time share is
around 0.39 rather than 0.30 - worth carrying into Part 8, because
`observed_caution_multiplier` is now returned by `calibrate_cautions` and
can be checked against this directly. If the observed multiplier comes back
near 1.45, this reconstruction is independent confirmation that the fix
addressed the actual fault. If it comes back near 2.0, something in the
chain is still not understood.

## Part 6 - decision 17, and where its failure actually shows

02a supersedes decision 17's redraw-until-non-overlapping, on the grounds
that conditioning the whole configuration on non-overlap reweights towards
shorter length vectors, "so lengths stop being marginally exponential".

That reasoning is sound in outline and the conclusion is right. The stated
diagnostic is not. Checking it properly matters, because decision 17 was
adopted to protect memorylessness and it should not be discarded on an
argument that fails the first check anyone runs against it.

In [8]:
target, mean_dur = 0.30, mean_dur_for(0.30)

alt = []
for s in seeds(1500):
    alt.extend(draw_alternating(np.random.default_rng(s), target, mean_dur)[1])
alt = np.array(alt)

rej, attempts, rej_shares = [], [], []
for s in seeds(1500):
    ep, n_att = draw_rejection(np.random.default_rng(s), target, mean_dur)
    if ep is None:
        continue
    attempts.append(n_att)
    rej_shares.append(share(ep))
    rej.extend(lengths_of(ep))
rej = np.array(rej)

pd.DataFrame([{
    "process":            name,
    "n_episodes":         len(x),
    "mean_length_s":      round(x.mean()),
    "claimed_mean_s":     round(mean_dur),
    "cv":                 round(cv(x), 3),
    # shape alone: does it look exponential with *its own* mean?
    "KS_p_vs_fitted_exp": round(stats.kstest(x, "expon",
                                             args=(0, x.mean())).pvalue, 3),
    # shape and scale: is it the exponential the model claims to draw?
    "KS_p_vs_claimed_exp": f"{stats.kstest(x, 'expon', args=(0, mean_dur)).pvalue:.2g}",
} for name, x in (("alternating", alt), ("rejection (dec 17)", rej))])

,process,n_episodes,mean_length_s,claimed_mean_s,cv,KS_p_vs_fitted_exp,KS_p_vs_claimed_exp
0,alternating,12543,3221,3240,0.995,0.969,0.95
1,rejection (dec 17),12000,2321,3240,0.985,0.696,8.5e-150


In [9]:
print(f"rejection sampling accepted 1 draw in {np.mean(attempts):.1f}")
print(f"realised share under rejection: {np.mean(rej_shares):.4f} "
      f"against a target of {target:.2f} "
      f"({(np.mean(rej_shares) - target) / target * 100:+.0f}%)")

rejection sampling accepted 1 draw in 10.8
realised share under rejection: 0.2149 against a target of 0.30 (-28%)


**The stated diagnostic does not fire.** Conditioned lengths have a cv of
1.00 and pass a Kolmogorov-Smirnov test against an exponential comfortably.
The marginal shape survives the conditioning intact.

**What the conditioning destroys is the scale.** The mean episode falls from
3240s to about 2300s - the distribution is still exponential, just a
different exponential from the one calibration asked for. Tested against the
exponential the model *claims* to draw, rather than one fitted to its own
output, it is rejected at a p-value with 170 zeros in it.

The consequence is worse than the merge it was competing with. Rejection
sampling lands a 0.30 target at 0.214, a 29% shortfall, against the merging
draw's 15%. Decision 17 traded a coverage error for a larger coverage error
while leaving the shape it was protecting untouched.

So: right decision, wrong reason, and the right reason is a stronger
argument. 02a's paragraph on this should be corrected rather than left to be
found - a reader who checks the cv will find 1.00 and conclude the
supersession was unjustified.

## Part 7 - the seed-noise caveat, and where it actually sits

02a's consequences note a standard deviation of roughly 0.07 on the realised
share with about eight episodes in a 24-hour race, and warns that decision
11's 50-seed sweep budget will be noisier than the 200-seed headline.

The warning is right and the number is worth pinning down, because the sd is
not a property of the engine - it is a property of the operating point. For
a fixed share, noise falls as episodes get shorter and more numerous, since
the race is averaging over more of them.

In [10]:
rows = []
for target in (0.18, 0.30, 0.407):
    for n_ep in (6, 8, 12, 16, 24):
        mean_dur = mean_dur_for(target, n_ep)
        sh = [share(draw_alternating(np.random.default_rng(s), target,
                                     mean_dur)[0])
              for s in seeds(4000)]
        rows.append({"time_share": target, "episodes": n_ep,
                     "mean_episode_s": round(mean_dur),
                     "sd_of_realised_share": round(np.std(sh), 4)})

noise = pd.DataFrame(rows)
noise.pivot(index="episodes", columns="time_share",
            values="sd_of_realised_share")

time_share,0.180,0.300,0.407
episodes,,,
6,0.0844,0.1176,0.1354
8,0.0733,0.1044,0.1172
12,0.0596,0.0846,0.0957
16,0.0520,0.0723,0.0856
24,0.0428,0.0598,0.0698


An sd of 0.07 with eight episodes corresponds to a time share of about 0.18.
But Part 5 puts the corrected Daytona share nearer 0.39, where eight
episodes gives an sd of about 0.12 - not 0.07 but close to double it.

Which of those the engine actually sits at is a Part 8 question, and the two
readings have different consequences for decision 11. At 0.07 a 50-seed
sweep carries a standard error of about 0.010 on the share; at 0.12 it is
0.017, and a sweep curve would need roughly three times the seeds to resolve
the same difference. Worth settling before a caution-sensitive sweep is read
as a trend, which is exactly what 02a warns about.

## Part 8 - against the engine

Everything above verifies the *model*. This part verifies that
`src/endurance` implements it, which is the half that would actually catch
a regression.

Two of the checks need the real timing data and the frozen dials notebook
01 writes. Where those are missing the cells say so and carry on, rather
than failing or - worse - quietly substituting something else.

In [11]:
import sys
from pathlib import Path


def find_project_root(marker: str = "src/endurance") -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker!r} at or above {here}.")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "tests"))

from endurance import Compat, RaceConfig, run_race          # noqa: E402
from endurance.engine import CautionTimeline                # noqa: E402

# The same locations notebook 01 uses, not a second convention.
PARAMS_DIR = ROOT / "data" / "processed"      # where 01 freezes the dials
DATA = ROOT / "data" / "raw" / "laps.csv"
FROZEN = PARAMS_DIR / "imsa.json"
print("project root:", ROOT)
print("frozen dials:", "found" if FROZEN.exists() else "not written yet")

project root: /Users/joshzola/Documents/motorsport/endurance racing/endurance strategy rl
frozen dials: found


### 8.1 The parity gate

02a promises that with every correction switched off the engine is the
engine notebook 01 validated, bit for bit. That promise is guarded by a
test rather than by this notebook, and the honest thing here is to run the
test rather than re-derive a weaker version of it.

The reference numbers in that test were taken off the engine *before* 02a
touched it. A test that only compared the new engine against itself would
pass whatever had broken.

In [12]:
try:
    from test_compat import (GOLDEN_01, fingerprint,        # noqa: E402
                             make_config as parity_config)
except ModuleNotFoundError as exc:
    # The suite is the canonical home of this check; the notebook only
    # reports it. No point failing the whole notebook over a dev dependency.
    print(f"parity gate needs {exc.name} - run `pytest tests/` for it")
    parity = None
else:
    parity = {seed: fingerprint(run_race(parity_config(), seed=seed,
                                         compat=Compat.v01())) == expected
              for seed, expected in GOLDEN_01.items()}
    print("Compat.v01() reproduces the pre-02a engine on every "
          "reference seed:", all(parity.values()))

parity

Compat.v01() reproduces the pre-02a engine on every reference seed: True


{0: True, 3: True, 7: True}

### 8.2 Does the engine draw the process Part 2 describes?

Part 2 has already established what the model does, so any disagreement
here is an implementation bug and not a modelling question. This runs
`CautionTimeline.draw` over the same seed count and the same operating
points as Part 2 and puts the two side by side.

In [13]:
rows = []
for target in (0.05, 0.18, 0.30):
    mean_dur = mean_dur_for(0.30)          # the same episode scale Part 2 used

    model_shares, model_pooled, model_counts = [], [], []
    eng_shares, eng_pooled, eng_counts = [], [], []

    for s in seeds(N_SEEDS):
        ep, drawn = draw_alternating(np.random.default_rng(s), target, mean_dur)
        model_shares.append(share(ep))
        model_pooled.extend(drawn)
        model_counts.append(len(ep))

        tl = CautionTimeline.draw(DURATION_S, target, mean_dur,
                                  np.random.default_rng(s))
        eng_shares.append(tl.total_caution_s() / DURATION_S)
        eng_pooled.extend(lengths_of(tl.periods))
        eng_counts.append(len(tl.periods))

    rows.append({"target": target,
                 "model_share": round(float(np.mean(model_shares)), 4),
                 "engine_share": round(float(np.mean(eng_shares)), 4),
                 "model_episodes": round(float(np.mean(model_counts)), 1),
                 "engine_episodes": round(float(np.mean(eng_counts)), 1),
                 "model_cv": round(cv(model_pooled), 3),
                 "engine_cv": round(cv(eng_pooled), 3)})

against_engine = pd.DataFrame(rows)
against_engine["share_gap"] = (against_engine["engine_share"]
                               - against_engine["model_share"]).round(4)
against_engine

,target,model_share,engine_share,model_episodes,engine_episodes,model_cv,engine_cv,share_gap
0,0.05,0.0502,0.0502,1.4,1.4,0.988,0.973,0.0
1,0.18,0.1813,0.1813,5.0,5.0,0.995,0.994,0.0
2,0.30,0.3022,0.3022,8.3,8.3,1.010,1.008,0.0


The two implementations were written from the same description by
different routes, so agreement to within Monte Carlo noise is the check
passing. The engine's episode lengths are clipped by the end of the race
where the model's are not, which pulls its pooled cv slightly below one -
that is censoring, not a different process, and Part 2 excludes the
clipped episodes for the same reason.

### 8.3 The observed caution multiplier, against the assumed one

`calibrate_cautions` returns `observed_caution_multiplier` alongside the
calibrated figures. Part 5 predicts it lands near 1.45. Reporting it next
to the assumed `caution_pace_multiplier` rather than substituting it is
the whole point: the assumed value stays in `ASSUMED_FIELDS` and gets
swept, and the observed one says how far the assumption sits from what the
data saw.

In [14]:
if not (DATA.exists() and FROZEN.exists()):
    print("no timing data or frozen dials - 8.3 and 8.4 need both, skipping")
else:
    from endurance import calibrate                          # noqa: E402

    con = calibrate.connect(str(DATA))
    for series_code, pattern in (("imsa", "%daytona%"), ("wec", "%le mans%")):
        cfg = RaceConfig.load(PARAMS_DIR / f"{series_code}.json")
        cls = cfg.classes[0]
        # Scoped to the edition the frozen dials came from, per 00's re-run.
        sid = calibrate.find_race(con, series_code, pattern)["session_id"]
        report = calibrate.calibrate_cautions(con, sid, cls.base_pace_s)
        print(f"{series_code}: observed multiplier "
              f"{report['observed_caution_multiplier']:.2f}  "
              f"assumed {cls.caution_pace_multiplier:.2f}  "
              f"share {report['caution_rate']:.3f}  "
              f"episodes {report['n_caution_episodes']}")

imsa: observed multiplier 1.87  assumed 1.60  share 0.353  episodes 9


wec: observed multiplier 1.83  assumed 1.60  share 0.088  episodes 6


### 8.4 Which operating point are we actually at?

Part 7's table is only useful once we know which row of it the calibrated
engine sits on, because that is what decides whether decision 11's sweep
budget is adequate.

In [15]:
if not (DATA.exists() and FROZEN.exists()):
    print("no timing data or frozen dials - skipping")
else:
    for series_code in ("imsa", "wec"):
        cfg = RaceConfig.load(PARAMS_DIR / f"{series_code}.json")
        cls = cfg.classes[0]
        n_ep = max(round(cls.caution_rate * cfg.duration_s
                         / cls.caution_mean_dur_s), 1)
        sd = np.std([share(draw_alternating(np.random.default_rng(s),
                                            cls.caution_rate,
                                            cls.caution_mean_dur_s,
                                            cfg.duration_s)[0])
                     for s in seeds(2000)])
        print(f"{series_code}: calibrated share {cls.caution_rate:.3f} over "
              f"about {n_ep} episodes -> sd of realised share {sd:.3f}; "
              f"a 50-seed sweep carries se {sd / np.sqrt(50):.4f}")

imsa: calibrated share 0.353 over about 9 episodes -> sd of realised share 0.105; a 50-seed sweep carries se 0.0149
wec: calibrated share 0.088 over about 6 episodes -> sd of realised share 0.046; a 50-seed sweep carries se 0.0065


## Part 9 - the four corrections, one at a time

Everything from here needs a race, so it needs dials. The frozen dials
notebook 01 writes are used where they exist; where they do not, a
stand-in field is built and labelled as one, so the notebook executes
either way and nobody mistakes the stand-in for calibration.

`Compat` is the switchboard. Each correction is a flag, and `Compat.v01()`
turns all of them off at once - that combination is the parity gate 8.1
just checked.

In [16]:
from endurance import ClassDials                             # noqa: E402

SEED_BANK = range(20)


def race_config():
    """The calibrated IMSA dials if 01 has written them, else a stand-in."""
    frozen = PARAMS_DIR / "imsa.json"
    if frozen.exists():
        return RaceConfig.load(frozen), True

    gtp = ClassDials(
        series_code="imsa", class_name="GTP", base_pace_s=97.5,
        deg_slope_s_per_lap=0.012, pace_spread_s=0.5, lap_noise_s=0.5,
        caution_rate=0.20, caution_mean_dur_s=600.0, green_stint_laps=30.0,
        fuel_per_lap=1 / 30, fuel_per_lap_caution=0.6 / 30, tyre_life_laps=60.0,
        pit_time_mean_s=47.0, pit_time_std_s=2.0, n_cars=8)
    gtd = ClassDials(
        series_code="imsa", class_name="GTD", base_pace_s=112.0,
        deg_slope_s_per_lap=0.020, pace_spread_s=0.9, lap_noise_s=0.6,
        caution_rate=0.20, caution_mean_dur_s=600.0, green_stint_laps=28.0,
        fuel_per_lap=1 / 28, fuel_per_lap_caution=0.6 / 28, tyre_life_laps=56.0,
        pit_time_mean_s=45.0, pit_time_std_s=2.0, n_cars=10)
    return RaceConfig(name="stand-in", series_code="imsa",
                      duration_s=24 * 3600.0, classes=[gtp, gtd]), False


CFG, CALIBRATED = race_config()
HEADLINE = CFG.classes[0].class_name
if not CALIBRATED:
    print("NOT CALIBRATED - stand-in field. Run notebook 01 first for "
          "numbers that mean anything about Daytona.")
print(f"{CFG.name}: {CFG.total_cars} cars, {CFG.duration_s / 3600:.0f}h, "
      f"headline class {HEADLINE}")

Daytona 24 2026: 60 cars, 24h, headline class GTD


In [17]:
STEPS = [
    ("01 engine",         Compat.v01()),
    ("+ per-car streams", Compat(legacy_pit=True, legacy_caution_pace=True,
                                 legacy_traffic=True)),
    ("+ pit layer",       Compat(legacy_caution_pace=True, legacy_traffic=True)),
    ("+ compression",     Compat(legacy_traffic=True)),
    ("+ traffic",         Compat()),
]

rows = []
for label, compat in STEPS:
    winners, spreads = [], []
    for seed in SEED_BANK:
        c = run_race(CFG, seed=seed, compat=compat).classification()
        c = c[c["class"] == HEADLINE]
        winners.append(c["laps"].max())
        spreads.append(c["laps"].max() - c["laps"].min())
    rows.append({"engine": label,
                 "winner_laps": round(float(np.mean(winners)), 1),
                 "sd_across_seeds": round(float(np.std(winners)), 1),
                 "class_spread_laps": round(float(np.mean(spreads)), 1)})

corrections = pd.DataFrame(rows)
corrections

,engine,winner_laps,sd_across_seeds,class_spread_laps
0,01 engine,690.4,23.9,8.2
1,+ per-car streams,664.0,27.6,9.2
2,+ pit layer,665.4,27.5,9.2
3,+ compression,708.8,19.7,9.6
4,+ traffic,708.7,20.0,9.2


Read down the last column rather than across the first. Compression is
doing nearly all of the work: it takes the headline class from finishing
strung out over several laps to finishing covered by about one, which is
where real endurance classifications sit and where notebook 01's did not.

The per-car streams cost the winner laps *and* widen the seed spread. That
is not a regression, it is the paired-comparison defect being paid off -
under 01 a change of strategy silently reshuffled every other car's noise,
and the stability that bought was not information.

## Part 10 - what each correction actually did

The table above is the summary. These are the mechanisms, one at a time,
because a summary nobody can take apart is a summary nobody should
believe.

### 10.1 Noise is a function of the seed, not of the strategy

Degradation, traffic and cautions are switched off here, so a green lap
time is exactly the car's pace plus that lap's noise. Two strategies that
pit at completely different moments must then produce identical lap times
at identical lap numbers. Under 01's shared generator they do not, and the
difference is of the same order as the strategic effects 02 sets out to
measure.

In [18]:
from endurance import (FixedLapStint, RunToFuelWindow,       # noqa: E402
                       scale_dials)

quiet = scale_dials(CFG, deg_slope_s_per_lap=0.0, traffic_penalty_s=0.0,
                    caution_rate=0.0)


def lap_times_by_lap(result, car_id):
    sub = result.laps[result.laps["car_id"] == car_id]
    return dict(zip(sub["lap"], sub["lap_time"]))


rows = []
for label, compat in (("01 engine", Compat.v01()), ("02a engine", Compat())):
    a = run_race(quiet, default_strategy=RunToFuelWindow(), seed=11,
                 compat=compat)
    b = run_race(quiet, default_strategy=FixedLapStint(stint_laps=7), seed=11,
                 compat=compat)
    car = a.laps["car_id"].iloc[0]
    la, lb = lap_times_by_lap(a, car), lap_times_by_lap(b, car)
    shared = sorted(set(la) & set(lb))
    diffs = np.array([abs(la[k] - lb[k]) for k in shared])
    rows.append({"engine": label, "laps_compared": len(shared),
                 "max_lap_time_difference_s": round(float(diffs.max()), 6),
                 "total_difference_s": round(float(diffs.sum()), 1)})

pd.DataFrame(rows)

,engine,laps_compared,max_lap_time_difference_s,total_difference_s
0,01 engine,781,2.242379,366.4
1,02a engine,817,0.000000,0.0


### 10.2 A stop has a shape, and the two series shape it differently

The pit layer is anchored to the measured mean: a full tank plus tyres
costs exactly `pit_time_mean_s`, in both series, by construction. The
level is data and stays data. Everything the layer says is about stops
that are *not* full service - the only place a rulebook can tell you
something lap timing cannot.

IMSA allows four over the wall including the refueller, air jack and tyre
changes (art. 34.1.1), so the jobs overlap and a stop costs the longer of
them. WEC forbids tools during the refuelling phase (art. 12), so they
queue and a stop costs their sum.

In [19]:
from endurance import PitRules, stop_cost                    # noqa: E402

dials = CFG.classes[0]
rows = []
for series_code in ("imsa", "wec"):
    rules = PitRules.for_series(series_code)
    for label, fuel, tyres in (("full tank + tyres", 1.0, True),
                               ("half tank + tyres", 0.5, True),
                               ("splash, no tyres", 0.3, False),
                               ("tyres only", 0.0, True)):
        rows.append({"series": series_code, "stop": label,
                     "cost_s": round(stop_cost(dials, rules, fuel, tyres), 1)})

shapes = pd.DataFrame(rows).pivot(index="stop", columns="series",
                                  values="cost_s")
shapes["01 engine"] = round(dials.pit_time_mean_s, 1)
shapes.reindex(["full tank + tyres", "half tank + tyres",
                "splash, no tyres", "tyres only"])

series,imsa,wec,01 engine
stop,,,
full tank + tyres,90.5,90.5,90.5
half tank + tyres,56.6,72.4,90.5
"splash, no tyres",43.0,33.5,90.5
tyres only,54.3,54.3,90.5


Every one of those cost `pit_time_mean_s` under notebook 01. The
splash-and-dash planner in 02's roster depends entirely on the difference
between these rows, which is why decision 4/7 promoted this layer from an
improvement to a prerequisite.

One consequence worth stating rather than discovering later: the baselines
never arrive with an empty tank, so turning the layer on makes even their
ordinary stops a little cheaper than 01 priced them. The saving is the
fuel that was never put in.

### 10.3 The pit lane closes, and reopens by rulebook

**A correction to the blueprint.** Section 7C states that the pit lane
closes under FCY in IMSA and stays open in WEC, and calls that a rulebook
fact rather than an assumption. The WEC half is wrong. Article 14.5.2
closes the pit entry when FCY is announced, leaving the exit open, and
article 14.6.5 closes the entry for the first three laps of a safety car,
two if it follows an FCY.

The real difference is not open against closed - it is *staged* against
*unstaged* reopening, plus IMSA's Short FCY, which never opens at all.
IMSA releases GTP and LMP2 on the first lap after the pits are declared
open and the GT classes on the next (art. 46.3.1); WEC releases everyone
together. That is still most of the reason to simulate both series, and it
means the caution gambler in 02's roster carries real risk in both.

In [20]:
from endurance import lane_status                            # noqa: E402


class _Episode:
    """A caution timeline with periods chosen rather than drawn."""

    def __init__(self, periods):
        self.periods = periods


CAUTION_LAP_S = min(c.base_pace_s * c.caution_pace_multiplier
                    for c in CFG.classes)
episode = _Episode([(3600.0, 3600.0 + 8 * CAUTION_LAP_S)])

rows = []
for laps_in in (0.5, 1.5, 2.5, 3.5, 4.5):
    t = 3600.0 + laps_in * CAUTION_LAP_S
    row = {"caution_laps_elapsed": laps_in}
    for series_code, cls_name in (("imsa", "GTP"), ("imsa", "GTD"),
                                  ("wec", "HYPERCAR")):
        st = lane_status(PitRules.for_series(series_code), episode, t, cls_name,
                         caution_lap_s=CAUTION_LAP_S,
                         duration_s=CFG.duration_s)
        row[f"{series_code} {cls_name}"] = "open" if st.open else "shut"
    rows.append(row)

pd.DataFrame(rows)

,caution_laps_elapsed,imsa GTP,imsa GTD,wec HYPERCAR
0,0.5,shut,shut,shut
1,1.5,shut,shut,shut
2,2.5,open,shut,shut
3,3.5,open,open,shut
4,4.5,open,open,open


### 10.4 Compression, and the lap a wave-around hands back

Behind a safety car everybody runs the safety car's lap, and a car with a
gap closes a share of it each lap until the field is queued up. Both are
applied as adjustments to *lap times*, never to the running order: 01's
central claim is that position is derived from accumulated race time and
never simulated, and compression is exactly the machinery that would break
it. A test rebuilds every car's finishing time from its own laps and
demands the classification agree.

Wave-arounds use the same eligibility rule in both books - a car whose
class leader is behind it in the queue (IMSA art. 46.2.2 and 46.4.1, WEC
art. 14.6.4) - which selects lapped cars on its own. IMSA runs it twice,
WEC once.

**One artefact to know about.** A wave-by is a timing-system credit, not
physics: the car is given a lap it did not drive. Expressed as a lap time
that is one very short caution lap, flagged `wave_by` in the lap record so
that nothing reading caution laps as pace picks it up. It is the least
physical thing in the engine.

In [21]:
rows = []
for label, compat in (("01 engine", Compat(legacy_caution_pace=True)),
                      ("02a engine", Compat())):
    result = run_race(CFG, seed=4, compat=compat)
    laps = result.laps
    caution = laps[laps["under_caution"] & ~laps["wave_by"]]
    by_class = caution.groupby("class")["lap_time"].mean()
    rows.append({
        "engine": label,
        "safety_car_lap_s": round(CAUTION_LAP_S, 1),
        **{f"{name}_caution_lap_s": round(float(v), 1)
           for name, v in by_class.items()},
        "wave_arounds": int(laps["wave_by"].sum()),
    })

pd.DataFrame(rows)

,engine,safety_car_lap_s,GTD_caution_lap_s,GTDPRO_caution_lap_s,GTP_caution_lap_s,LMP2_caution_lap_s,wave_arounds
0,01 engine,156.9,173.6,173.0,156.8,163.5,0
1,02a engine,156.9,143.4,144.3,154.0,146.4,242


Under 01 a GTD caution lap was 1.6 times a *GTD* lap, so the class that
was slow under green stayed slow under yellow, the field kept its shape,
and a caution cost nothing positionally and gained nothing. That left the
assumed `pit_caution_discount` carrying the entire caution story on its
own. It carries a good deal less of it now.

### 10.5 Traffic that a stop can do something about

Under 01 a car counted as an obstruction if its *base* pace was slower
than yours, so the same cars blocked you on lap two and lap two hundred
and no stop could change it. "I will come out into traffic if I stop now"
was not merely unimplemented, it was unrepresentable.

It is representable now, and the honest report is that it does very
little. A field all running one plan pits in lockstep, so every car
carries the same tyre age, degradation cancels out of the comparison, and
the correction changes nothing whatsoever. Once the stints are staggered
it starts to bite - and even then it is worth tenths against class gaps
worth seconds.

In [22]:
car_ids = [f"{cls.class_name}-{j + 1:02d}"
           for cls in CFG.classes for j in range(cls.n_cars)]
plans = {cid: (FixedLapStint(stint_laps=10 + 3 * (i % 4)) if i % 2
               else RunToFuelWindow())
         for i, cid in enumerate(car_ids)}

rows = []
for label, strategies in (("one plan for everyone", None),
                          ("stints staggered", plans)):
    counts = {}
    for engine, compat in (("01 engine", Compat(legacy_traffic=True)),
                           ("02a engine", Compat())):
        r = run_race(CFG, strategies, seed=3, compat=compat)
        counts[engine] = int(r.laps["blockers"].sum())
    rows.append({"field": label, **counts,
                 "difference": counts["02a engine"] - counts["01 engine"]})

pd.DataFrame(rows)

,field,01 engine,02a engine,difference
0,one plan for everyone,21527,21626,99
1,stints staggered,21928,22062,134


## Part 11 - notebook 01's Part 6, and the units error in it

02a flags that 01's validation table compares a simulated caution *time*
share against a real caution *lap* share. Both sides should be time
shares. The correction is reported before as well as after, because Part 6
is one of the few places the units error was visible at all.

Notebook 01 carries the corrected table; this is the working that
justifies changing it, and part of the reason 01's numbers moved.

In [23]:
if not (DATA.exists() and FROZEN.exists()):
    print("no timing data or frozen dials - this check needs both, skipping")
else:
    from endurance import calibrate                          # noqa: E402

    con = calibrate.connect(str(DATA))
    rows = []
    for series_code, pattern in (("imsa", "%daytona%"), ("wec", "%le mans%")):
        sid = calibrate.find_race(con, series_code, pattern)["session_id"]
        real = con.execute(
            "SELECT "
            "SUM(CASE WHEN flags IN ('FCY','SF','RF') THEN 1 ELSE 0 END) "
            "* 1.0 / COUNT(*), "
            "SUM(CASE WHEN flags IN ('FCY','SF','RF') THEN lap_time ELSE 0 END) "
            "/ SUM(lap_time) "
            "FROM laps "
            f"WHERE series_code = '{series_code}' "
            f"AND session_id = {sid} "
            "AND session = 'race' AND pit_time IS NULL "
            "AND flags IN ('GF','FCY','SF','RF')").fetchone()
        cfg = RaceConfig.load(PARAMS_DIR / f"{series_code}.json")
        sim = run_race(cfg, seed=0)
        rows.append({
            "series": series_code,
            "real_lap_share_what_01_compared": round(float(real[0]), 3),
            "real_time_share_what_it_should_be": round(float(real[1]), 3),
            "sim_time_share": round(
                sim.cautions.total_caution_s() / cfg.duration_s, 3)})
    units = pd.DataFrame(rows)
    print(units.to_string(index=False))

series  real_lap_share_what_01_compared  real_time_share_what_it_should_be  sim_time_share
  imsa                            0.239                              0.353           0.318
   wec                            0.046                              0.075           0.060


## Where this leaves us

**The stage gate.** All five conditions hold. `Compat.v01()` reproduces
the pre-02a engine bit for bit against reference numbers taken off it
before any of this was written; per-car lap noise is identical under two
strategies on one seed; compression changes no car's position except
through lap times; the caution-timeline independence test still passes;
and the suite is green, having grown from 42 tests to 92 with none of the
original 42 removed.

**Findings this stage produced that no decision document contains.**

1. *The blueprint is wrong about the WEC pit lane.* Section 7C calls it a
   rulebook fact that the lane stays open in WEC. It closes (arts. 14.5.2
   and 14.6.5). The real asymmetry is staged against unstaged reopening.
   That needs correcting in the blueprint rather than working around here,
   because 02b's benchmark must exclude closed windows in *both* series.

2. *Compression is doing nearly all the work.* Of the four corrections it
   is the one that moves the classification, and it moves it a long way -
   from a headline class strung out over several laps to one covered by
   about a lap. That is the direction real classifications sit in. Whether
   the magnitude is right is a validation question needing the real event
   rather than an argument, and it is the first thing 02b should settle
   before trusting a benchmark built on this engine.

3. *The traffic correction is necessary and nearly inert.* It fixes a real
   defect - stop timing could not affect traffic at all - but on a field
   running one strategy it changes nothing by construction, and on a
   staggered field it is worth tenths. Any 02 comparison run against a
   single-strategy background field will see none of it. By decision 14
   that is a finding rather than a bug, and it belongs in the write-up.

4. *A wave-around is bookkeeping, not physics.* It is implemented as a lap
   time so that position stays derived, and that lap time is not a lap
   anybody drives. It is flagged in the record; anything reading caution
   lap times as pace must exclude it.

5. The findings from Parts 1 to 7 stand: the seed-noise caveat is pinned
   to an operating point 8.4 locates, decision 17's stated failure does
   not reproduce, and its real failure is worse than the stated one.

**What 02a deliberately did not do.** No strategies and no benchmark -
that is the stage boundary. `pit_caution_discount` survives as an assumed
dial even though compression now does part of its job, because retiring it
is a calibration decision and this was not a calibration stage. The
assumed dials grew from five to ten, and every new one is in
`ASSUMED_FIELDS` and swept rather than trusted.

**Next.** 02b, the per-race benchmark - with the closed-window constraint
in both series, and with compression widening the gap between time-optimal
and position-optimal plans, which decision 5 already warns means *k* needs
to be generous.